# Correlation Analysis

In this notebook we show that the classification accuracy on the validation set correlates strongly with the reconstruction quality of the prototype samples (anchor_loss). This is an important finding for the practicality of RERC-based systems because without a means to pick the best classifier from purely unsupervised signals it would not be possible to know which of the hyperparameter-combinations should be selected for the final model. 

We show this later in the paper, so we only consider EMNIST digits and EMNIST balanced henceforth. 

In [19]:
import os
import sys

import pandas as pd
from pathlib import Path
from scipy.stats import pearsonr, spearmanr

#       {  Location of this notebook }
#           vvvvvvvvvvvvvvvv
SWEEP_DIR = Path().resolve().parent / 'results' / 'ssl_sweeps'

for sweep in SWEEP_DIR.rglob('*.csv'):
    print(f"Processing sweep: {sweep}")
    df = pd.read_csv(sweep)
    df = df[["val/accuracy", "train/anchor_loss"]].dropna()
    print(df.columns)
    corr, p_value = spearmanr(df["val/accuracy"], df["train/anchor_loss"])
    print(corr, p_value)
    print("-"*50)

Processing sweep: /home/thallybu/workspace/26ICDAR-RERC/results/ssl_sweeps/sweep_SelfSupervised_EMNIST_Digits_Conv.csv
Index(['val/accuracy', 'train/anchor_loss'], dtype='object')
-0.7770044185601394 5.123430193141729e-82
--------------------------------------------------
Processing sweep: /home/thallybu/workspace/26ICDAR-RERC/results/ssl_sweeps/sweep_SelfSupervised_EMNIST_Digits_DINO.csv
Index(['val/accuracy', 'train/anchor_loss'], dtype='object')
-0.7470140190005221 1.447829955480565e-72
--------------------------------------------------
Processing sweep: /home/thallybu/workspace/26ICDAR-RERC/results/ssl_sweeps/sweep_SelfSupervised_EMNIST_Digits_CLIP.csv
Index(['val/accuracy', 'train/anchor_loss'], dtype='object')
-0.5793789287843419 1.356258361591303e-37
--------------------------------------------------
Processing sweep: /home/thallybu/workspace/26ICDAR-RERC/results/ssl_sweeps/sweep_SelfSupervised_EMNIST_Balanced_Dino.csv
Index(['val/accuracy', 'train/anchor_loss'], dtype='object')

In [20]:
"""
Plotly scatter + regression: two subplots (Digits | Balanced), one trace per encoder.
"""
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from pathlib import Path
from scipy.stats import pearsonr, linregress

SWEEP_DIR = Path().resolve().parent / 'results' / 'ssl_sweeps'

COLORS = {"Conv": "#636EFA", "DINO": "#EF553B", "CLIP": "#00CC96"}

FILES = {
    "Digits": {
        "Conv": "sweep_SelfSupervised_EMNIST_Digits_Conv.csv",
        "DINO": "sweep_SelfSupervised_EMNIST_Digits_DINO.csv",
        "CLIP": "sweep_SelfSupervised_EMNIST_Digits_CLIP.csv",
    },
    "Balanced": {
        "Conv": "sweep_SelfSupervised_EMNIST_Balanced_Conv.csv",
        "DINO": "sweep_SelfSupervised_EMNIST_Balanced_Dino.csv",
        "CLIP": "sweep_SelfSupervised_EMNIST_Balanced_CLIP.csv",
    },
}

fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=["EMNIST Digits", "EMNIST Balanced"],
    shared_yaxes=False,
    horizontal_spacing=0.08,
)

for col_idx, (dataset, encoders) in enumerate(FILES.items(), start=1):
    x_all, y_all = [], []
    for encoder, fname in encoders.items():
        csv_path = SWEEP_DIR / fname
        df = pd.read_csv(csv_path)[["val/accuracy", "train/anchor_loss"]].dropna()

        x = df["train/anchor_loss"].values
        y = df["val/accuracy"].values
        y = np.log(y + 1e-8)  # Add small epsilon to avoid log(0)
        x_all.extend(x)
        y_all.extend(y)

        corr, _ = spearmanr(x, y)
        slope, intercept, *_ = linregress(x, y)
        # normalise label casing
        enc_label = encoder if encoder != "Dino" else "DINO"
        color = COLORS.get(enc_label, COLORS.get(encoder, "#AB63FA"))

        # scatter
        fig.add_trace(go.Scatter(
            x=x, y=y, mode="markers",
            name=f"{enc_label} {dataset} (r={corr:.2f})",
            marker=dict(color=color, size=5, opacity=0.4,
                        line=dict(width=0.3, color="white")),
            legendgroup=f"{enc_label}_{dataset}",
        ), row=1, col=col_idx)

        # regression line
        x_fit = np.linspace(x.min(), x.max(), 200)
        y_fit = slope * x_fit + intercept
        fig.add_trace(go.Scatter(
            x=x_fit, y=y_fit, mode="lines",
            line=dict(color=color, width=2),
            legendgroup=f"{enc_label}_{dataset}",
            showlegend=False,
        ), row=1, col=col_idx)

    # tighten axes to data with small padding
    x_arr, y_arr = np.array(x_all), np.array(y_all)
    x_pad = (x_arr.max() - x_arr.min()) * 0.04
    y_pad = (y_arr.max() - y_arr.min()) * 0.04
    fig.update_xaxes(range=[x_arr.min() - x_pad, x_arr.max() + x_pad],
                     title_text="Train Anchor Loss", row=1, col=col_idx)
    fig.update_yaxes(range=[y_arr.min() - y_pad, y_arr.max() + y_pad],
                     title_text="Validation Accuracy" if col_idx == 1 else "",
                     row=1, col=col_idx)

fig.update_layout(
    template="plotly_white",
    legend=dict(font=dict(size=10)),
    width=1050, height=480,
    margin=dict(t=45, b=50, l=55, r=20),
)

out = SWEEP_DIR.parent / "scatter_regression_plotly.html"
fig.write_html(out, include_plotlyjs="cdn")
fig.show()
print(f"Saved → {out}")

Saved → /home/thallybu/workspace/26ICDAR-RERC/results/scatter_regression_plotly.html


In [23]:
"""
Matplotlib scatter + regression: two subplots (Digits | Balanced).
Exports PDF and PGF (TikZ) for LaTeX inclusion.

Usage in LaTeX:
  \\usepackage{pgf}
  \\input{scatter_regression.pgf}
Or:
  \\includegraphics{scatter_regression.pdf}
"""
import matplotlib
matplotlib.use("pgf")  # must be before pyplot import

import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
from pathlib import Path
from scipy.stats import pearsonr, linregress

# ---------- PGF / LaTeX config ----------
matplotlib.rcParams.update({
    "pgf.texsystem": "pdflatex",
    "font.family": "serif",
    "font.serif": [],                # Computer Modern
    "text.usetex": True,
    "pgf.rcfonts": False,
    "axes.labelsize": 9,
    "font.size": 8,
    "legend.fontsize": 7,
    "xtick.labelsize": 7,
    "ytick.labelsize": 7,
})

# ---------- paths ----------
SWEEP_DIR = Path().resolve().parent / "results" / "ssl_sweeps"  # ← adjust

COLORS = {"Conv": "#1f77b4", "DINO": "#d62728", "CLIP": "#2ca02c"}

FILES = {
    "EMNIST Digits": {
        "Conv": "sweep_SelfSupervised_EMNIST_Digits_Conv.csv",
        "DINO": "sweep_SelfSupervised_EMNIST_Digits_DINO.csv",
        "CLIP": "sweep_SelfSupervised_EMNIST_Digits_CLIP.csv",
    },
    "EMNIST Balanced": {
        "Conv": "sweep_SelfSupervised_EMNIST_Balanced_Conv.csv",
        "DINO": "sweep_SelfSupervised_EMNIST_Balanced_Dino.csv",
        "CLIP": "sweep_SelfSupervised_EMNIST_Balanced_CLIP.csv",
    },
}

# textwidth ≈ 7in for two-column templates; use full width for two subplots
fig, axes = plt.subplots(1, 2, figsize=(7.0, 2.8), sharey=False)

for ax, (dataset, encoders) in zip(axes, FILES.items()):
    x_all, y_all = [], []
    for encoder, fname in encoders.items():
        csv_path = SWEEP_DIR / fname
        df = pd.read_csv(csv_path)[["val/accuracy", "train/anchor_loss"]].dropna()

        x = df["train/anchor_loss"].values
        y = df["val/accuracy"].values
        y = np.log(y + 1e-8)  # Add small epsilon to avoid log(0)
        x_all.extend(x) 
        y_all.extend(y)

        corr, _ = spearmanr(x, y)
        slope, intercept, *_ = linregress(x, y)
        enc_label = encoder if encoder != "Dino" else "DINO"
        color = COLORS.get(enc_label, COLORS.get(encoder, "#9467bd"))

        label = rf"{enc_label} ($\rho_s={corr:.2f}$)"
        ax.scatter(x, y, c=color, s=6, alpha=0.30,
                   edgecolors="none", rasterized=True)

        x_fit = np.linspace(x.min(), x.max(), 200)
        y_fit = slope * x_fit + intercept
        ax.plot(x_fit, y_fit, color=color, linewidth=1.3, label=label)

    # tighten to data
    x_arr, y_arr = np.array(x_all), np.array(y_all)
    x_pad = (x_arr.max() - x_arr.min()) * 0.04
    y_pad = (y_arr.max() - y_arr.min()) * 0.04
    ax.set_xlim(x_arr.min() - x_pad, 0.08) # x_arr.max() + x_pad)
    ax.set_ylim(y_arr.min() - y_pad, y_arr.max() + y_pad)

    ax.set_xlabel("Train Anchor Loss")
    ax.set_title(dataset, fontsize=9)
    ax.legend(loc="best", frameon=True, framealpha=0.9, edgecolor="0.8")
    ax.grid(True, linewidth=0.3, alpha=0.5)

axes[0].set_ylabel(r"$\log(\mathrm{Acc})$")

fig.tight_layout(pad=0.4)

out_dir = SWEEP_DIR.parent
fig.savefig(out_dir / "scatter_regression.pdf")
fig.savefig(out_dir / "scatter_regression.pgf")
print(f"Saved PDF → {out_dir / 'scatter_regression.pdf'}")
print(f"Saved PGF → {out_dir / 'scatter_regression.pgf'}")

Saved PDF → /home/thallybu/workspace/26ICDAR-RERC/results/scatter_regression.pdf
Saved PGF → /home/thallybu/workspace/26ICDAR-RERC/results/scatter_regression.pgf
